# 03 - Modelo de Clustering

Entrenamiento y seleccion del modelo. El flujo es:

1. Separar los productos sin movimiento.
2. Escalar y aplicar PCA.
3. Buscar el numero optimo de clusters (Elbow + Silhouette + Davies-Bouldin).
4. Entrenar KMeans, Jerarquico Aglomerativo y DBSCAN.
5. Comparar con metricas internas y elegir el ganador.

---

In [ ]:
# Permite importar el paquete src desde la carpeta notebooks/
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config as cfg
from src import utils

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
utils.aplicar_estilo_graficas()

# En notebook queremos ver las figuras en linea, no solo guardarlas en disco
%matplotlib inline

print("Raiz del proyecto:", RAIZ)
print("Archivo fuente existe:", cfg.ARCHIVO_EXCEL.exists())

In [ ]:
from src import feature_engineering as fe
from src import clustering as cl

features = fe.cargar_features()
print(f"Features: {features.shape[0]:,} filas x {features.shape[1]} columnas")

## 1. Separacion de los productos sin movimiento

Explicado en el notebook 02: son un punto identico repetido cientos de veces.
Si entraran al modelo inflarian las metricas de calidad y desplazarian los
centroides de los grupos reales.

In [ ]:
con_mov, sin_mov = cl.separar_sin_movimiento(features)

## 2. Escalado

Dos decisiones importantes.

### 2.1 `log1p` sobre las variables de volumen

`total_unidades` va de 0 a mas de 68.000, con mediana alrededor de 272. Es una
distribucion con cola muy larga.

**Por que importa.** KMeans minimiza distancia euclidiana. Con una variable tan
sesgada, incluso despues de estandarizar, los pocos productos estrella quedan a
varias desviaciones del resto y **absorben la particion**: el modelo gasta sus
clusters separando "un top-seller" de "todo lo demas" en lugar de encontrar
patrones de comportamiento utiles.

`log1p` comprime la cola y deja que la distancia refleje diferencias relativas
(el doble de venta) en lugar de absolutas (10.000 unidades mas).

### 2.2 Las dummies no se re-escalan

Ya viven en 0/1, un rango comparable al de las numericas estandarizadas.
Estandarizarlas dividiria por su desviacion, que es pequena en los niveles poco
frecuentes: el resultado seria que "pertenecer a una categoria rara" pesaria mas
que todo el comportamiento de venta junto.

In [ ]:
# Efecto del log1p en la asimetria de las variables de volumen
from scipy import stats

print("Asimetria (skewness) antes y despues de log1p:")
for columna in cl.VARIABLES_LOG:
    antes = stats.skew(con_mov[columna])
    despues = stats.skew(np.log1p(con_mov[columna]))
    print(f"  {columna:<20} antes = {antes:>7.2f}   despues = {despues:>6.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(con_mov["total_unidades"], bins=50, color="indianred", edgecolor="white")
axes[0].set_title("total_unidades sin transformar\n(cola larga: pocos productos dominan)")
axes[0].set_xlabel("Unidades")
axes[1].hist(np.log1p(con_mov["total_unidades"]), bins=50, color="seagreen", edgecolor="white")
axes[1].set_title("log1p(total_unidades)\n(distribucion utilizable para distancia euclidiana)")
axes[1].set_xlabel("log(1 + unidades)")
plt.show()

In [ ]:
matriz, columnas_modelo, scaler = cl.construir_matriz(con_mov)
print(f"\nMatriz del modelo: {matriz.shape}")
print(f"Variables: {columnas_modelo}")

## 3. PCA: reduccion de dimensionalidad

Se conserva el 90% de la varianza. El PCA cumple dos funciones aqui:

1. **Resuelve la redundancia** entre las variables de volumen, que en el EDA
   aparecian muy correlacionadas. Sin PCA, el volumen contaria dos veces en la
   distancia.
2. **Mitiga la maldicion de la dimensionalidad** que introducen las 18 columnas
   One-Hot. En espacios de alta dimension las distancias euclidianas se
   homogenizan y todo empieza a parecer equidistante, lo que degrada cualquier
   algoritmo basado en distancia.

In [ ]:
matriz_pca, pca = cl.aplicar_pca(matriz)

varianza_acum = np.cumsum(pca.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1),
            pca.explained_variance_ratio_ * 100, color="steelblue")
axes[0].set_title("Varianza explicada por componente")
axes[0].set_xlabel("Componente principal")
axes[0].set_ylabel("% de varianza")

axes[1].plot(range(1, len(varianza_acum) + 1), varianza_acum * 100, marker="o", color="darkblue")
axes[1].axhline(cfg.VARIANZA_PCA * 100, color="red", linestyle="--",
                label=f"Objetivo {cfg.VARIANZA_PCA:.0%}")
axes[1].set_title("Varianza acumulada")
axes[1].set_xlabel("Numero de componentes")
axes[1].set_ylabel("% acumulado")
axes[1].legend()
plt.show()

print(f"Componentes retenidas: {matriz_pca.shape[1]} de {matriz.shape[1]} variables originales")

### Que representa cada componente

Vale la pena mirar los pesos: ayudan a interpretar despues el scatter de clusters.

In [ ]:
cargas = pd.DataFrame(pca.components_[:3].T, index=columnas_modelo,
                      columns=["PC1", "PC2", "PC3"]).round(3)
print("Variables con mayor peso en las 3 primeras componentes:\n")
for componente in ["PC1", "PC2", "PC3"]:
    top = cargas[componente].abs().nlargest(5).index
    print(f"{componente} ({pca.explained_variance_ratio_[int(componente[-1]) - 1]:.1%} de varianza):")
    for variable in top:
        print(f"    {variable:<38} {cargas.loc[variable, componente]:>7.3f}")
    print()

## 4. Numero optimo de clusters

Se calculan tres criterios para k de 2 a 12:

- **Elbow (inercia)**: donde deja de ganarse compacidad al agregar grupos.
- **Silhouette**: que tan bien separado y cohesionado esta cada grupo. Mayor es mejor.
- **Davies-Bouldin**: relacion entre dispersion interna y separacion. Menor es mejor.

In [ ]:
metricas_k = cl.evaluar_k(matriz_pca)
metricas_k.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

axes[0].plot(metricas_k["k"], metricas_k["inercia"], marker="o", color="darkblue")
k_codo = cl.k_por_codo(metricas_k)
axes[0].axvline(k_codo, color="red", linestyle="--", label=f"Codo detectado: k={k_codo}")
axes[0].set_title("Metodo del codo (inercia)")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inercia"); axes[0].legend()

axes[1].plot(metricas_k["k"], metricas_k["silhouette"], marker="o", color="seagreen")
k_sil = int(metricas_k.loc[metricas_k["silhouette"].idxmax(), "k"])
axes[1].axvline(k_sil, color="red", linestyle="--", label=f"Maximo: k={k_sil}")
axes[1].set_title("Silhouette Score (mayor es mejor)")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette"); axes[1].legend()

axes[2].plot(metricas_k["k"], metricas_k["davies_bouldin"], marker="o", color="indianred")
k_db = int(metricas_k.loc[metricas_k["davies_bouldin"].idxmin(), "k"])
axes[2].axvline(k_db, color="red", linestyle="--", label=f"Minimo: k={k_db}")
axes[2].set_title("Davies-Bouldin (menor es mejor)")
axes[2].set_xlabel("k"); axes[2].set_ylabel("Davies-Bouldin"); axes[2].legend()

plt.show()

### Los criterios no coinciden: como se decide

Es el caso habitual en datos reales. Aqui el punto clave es la **amplitud de la
curva de Silhouette**: si el mejor y el peor k difieren en muy poco (menos de
0,05), esa metrica no esta distinguiendo nada real y su maximo cae casi siempre
en k=2, que para el negocio equivale a "vende mucho / vende poco". Con dos grupos
no se pueden diferenciar politicas de compra.

En ese escenario `recomendar_k` **prioriza el metodo del codo**, que si identifica
donde deja de ganarse compacidad. Si la curva de Silhouette tuviera un maximo
marcado, mandaria el puntaje combinado.

Ademas se aplica un piso operativo de 3 grupos.

In [ ]:
k_optimo, metricas_k = cl.recomendar_k(metricas_k)
print(f"\nk seleccionado: {k_optimo}")
print(f"Amplitud de la curva Silhouette: "
      f"{metricas_k['silhouette'].max() - metricas_k['silhouette'].min():.4f}")

## 5. Entrenamiento de los tres algoritmos

### 5.1 KMeans

Particiona minimizando la distancia al centroide. Es el modelo de referencia:
rapido, con centroides interpretables y sin dejar productos sin clasificar, lo
cual es un requisito practico (se necesita una recomendacion para cada SKU).

In [ ]:
etq_kmeans, modelo_kmeans = cl.entrenar_kmeans(matriz_pca, k_optimo)
print("Registros por cluster:", np.bincount(etq_kmeans))

### 5.2 Jerarquico Aglomerativo (enlace Ward)

Fusiona iterativamente los grupos mas cercanos. Su ventaja es el **dendrograma**:
muestra la estructura de anidamiento y permite ver si el corte en k grupos es
natural o forzado.

In [ ]:
etq_jerarquico, modelo_jerarquico = cl.entrenar_jerarquico(matriz_pca, k_optimo)
print("Registros por cluster:", np.bincount(etq_jerarquico))

In [ ]:
matriz_enlace, _ = cl.calcular_linkage(matriz_pca)
fig, ax = plt.subplots(figsize=(14, 6))
cl.graficar_dendrograma(matriz_enlace, k_optimo, ax=ax)
plt.show()

### 5.3 DBSCAN

Agrupa por densidad y marca como ruido los puntos aislados. No necesita que se le
diga cuantos grupos hay, pero si necesita `eps` (el radio de vecindad).

**Calibracion automatica de `eps`.** Se recorre la curva de k-distancias por
percentiles de menor a mayor: un `eps` pequeno marca casi todo como ruido y, a
medida que crece, el ruido baja. Se toma el primer valor que deja el ruido por
debajo del 10% y produce al menos 2 grupos.

In [ ]:
min_samples = int(np.clip(2 * matriz_pca.shape[1], 5, 20))
distancias = cl.curva_k_distancias(matriz_pca, min_samples)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(distancias, color="darkorange")
ax.set_title(f"Curva de k-distancias (k = {min_samples})\n"
             "El codo marca la escala de densidad natural de los datos")
ax.set_xlabel("Registros ordenados por distancia")
ax.set_ylabel(f"Distancia al vecino {min_samples}")
plt.show()

etq_dbscan, modelo_dbscan = cl.entrenar_dbscan(matriz_pca)
print(f"\nClusters formados: {len(set(etq_dbscan) - {-1})}")
print(f"Registros marcados como ruido: {(etq_dbscan == -1).sum()} "
      f"({(etq_dbscan == -1).mean():.2%})")

## 6. Comparacion y seleccion del modelo

Tres metricas internas. En DBSCAN el ruido se excluye del calculo: no es un grupo,
y dejarlo dentro penalizaria injustamente el modelo.

In [ ]:
etiquetas = {"KMeans": etq_kmeans, "Jerarquico_Ward": etq_jerarquico, "DBSCAN": etq_dbscan}
metricas_modelos = cl.comparar_modelos(matriz_pca, etiquetas)
metricas_modelos

In [ ]:
ganador = cl.seleccionar_mejor_modelo(metricas_modelos)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
panel = [("silhouette", "Silhouette (mayor mejor)", "seagreen"),
         ("calinski_harabasz", "Calinski-Harabasz (mayor mejor)", "steelblue"),
         ("davies_bouldin", "Davies-Bouldin (menor mejor)", "indianred")]
for ax, (columna, titulo, color) in zip(axes, panel):
    ax.bar(metricas_modelos["modelo"], metricas_modelos[columna], color=color)
    ax.set_title(titulo)
    ax.tick_params(axis="x", rotation=20)
plt.show()

### Criterio de seleccion

Se elige por **mayor Silhouette entre los modelos que dejan menos del 10% de
ruido**. El filtro de ruido no es cosmetico: para planear compras se necesita una
recomendacion para cada SKU. Un modelo con mejor metrica que deja el 30% de los
productos sin clasificar no sirve operativamente.

**Lectura del resultado.** Los valores de Silhouette rondan 0,20. Es bajo en
terminos absolutos y conviene decirlo con claridad: **los productos no forman
grupos perfectamente separados**, forman un continuo. Eso es normal cuando se
mezclan variables de comportamiento con dummies de atributos, y no invalida el
modelo para su proposito. Lo que se necesita no es una particion matematicamente
perfecta, sino segmentos **estables e interpretables** a los que se pueda aplicar
una politica de compra distinta. El notebook 04 verifica que los grupos
efectivamente se diferencian en las variables que importan.

## 7. Ejecucion del paso completo

`clustering.ejecutar()` corre todo lo anterior, asigna el cluster a cada registro
y renumera los grupos de mayor a menor volumen (para que el cluster 0 sea siempre
el de mayor rotacion y el reporte sea estable entre ejecuciones).

In [ ]:
resultado = cl.ejecutar(features)

print(f"\nModelo ganador: {resultado.modelo_ganador}")
print(f"k optimo: {resultado.k_optimo}")
print(f"Componentes PCA: {resultado.matriz_pca.shape[1]}")

In [ ]:
# Perfil rapido: los clusters se diferencian en las variables del modelo?
resultado.df.groupby("Cluster")[cfg.FEATURES_NUMERICAS].mean().round(2)

### Verificacion

Las medias por cluster son claramente distintas en `promedio_mensual`,
`cobertura_meses` y `tendencia`. Aunque el Silhouette sea moderado, **los grupos
si capturan comportamientos de venta diferenciados**, que es lo que se necesita
para asignarles politicas de compra distintas.

**Siguiente paso:** `04_Resultados.ipynb`, donde se nombran e interpretan los
clusters y se construye la recomendacion de compra.